# Oracle ranking with recording first N seconds of performance metrics

In [1]:
import csv
import math
import re
from pathlib import Path
from collections import defaultdict
from contextlib import redirect_stdout
from io import StringIO

# ===== Paths (H100 only) =====
SPEC_ROOT_DIR = Path("/home/ac.zzheng/power/GPGPU/data/H100/spec_power_motif")
ML_ROOT_DIR   = Path("/home/ac.zzheng/power/GPGPU/data/H100/ml_power_motif")
TXT_OUT_PATH  = Path("/home/ac.zzheng/power/GPGPU/data/H100/h100_rank_results.txt")

SPEC_APPS = ["cloverleaf", "lbm", "minisweep", "miniweather", "pot3d", "tealeaf"]
ML_APPS = None  # None -> auto-discover

PROFILE_SECONDS = 15.0
ACTIVE_POWER_TH = 120.0
ACTIVE_SM_TH = 400.0

FILE_RE = re.compile(r"(?P<cap>\d+)_(?P<gpus>\d+)_gpu_metrics\.csv$")


def f(x):
    try:
        return float(str(x).strip())
    except Exception:
        return None


def mean(xs):
    return sum(xs) / len(xs) if xs else float("nan")


def fmt2(x):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return "nan"
    return f"{x:.2f}"


def list_apps(root, apps):
    if apps is not None:
        return apps
    if not root.exists():
        return []
    return sorted([p.name for p in root.iterdir() if p.is_dir()])


def detect_metric_keys(app_dir):
    keys = set()
    for p in app_dir.iterdir():
        m = FILE_RE.match(p.name)
        if m:
            keys.add((int(m.group("cap")), int(m.group("gpus"))))
    return keys


def read_runtime_lookup_spec(app_dir):
    """
    Build lookup {(total_cap, gpu_count): runtime_seconds}
    Auto-detect whether runtime.csv power_cap is total-cap or per-gpu-cap by overlap with metric keys.
    """
    out = {}
    p = app_dir / "runtime.csv"
    if not p.exists():
        return out

    metric_keys = detect_metric_keys(app_dir)
    rows = []

    with p.open(newline="") as fh:
        r = csv.DictReader(fh)
        cols = set(r.fieldnames or [])
        req = {"power_cap", "gpu_count", "runtime_seconds"}
        if not req.issubset(cols):
            return out
        for row in r:
            cap = f(row.get("power_cap"))
            g = f(row.get("gpu_count"))
            v = f(row.get("runtime_seconds"))
            if cap is None or g is None or v is None:
                continue
            g = int(round(g))
            rows.append((cap, g, v))

    # candidate A: cap already total
    raw = {(int(round(cap)), g): v for cap, g, v in rows}
    # candidate B: cap is per-gpu, convert to total
    per = {(int(round(cap * g)), g): v for cap, g, v in rows}

    raw_hits = sum(1 for k in raw if k in metric_keys)
    per_hits = sum(1 for k in per if k in metric_keys)

    return raw if raw_hits >= per_hits else per


def read_throughput_lookup_ml(app_dir):
    """
    Build lookup {(total_cap, gpu_count): throughput}
    Supports throughput.csv columns:
      - total_gpu_cap (preferred)
      - power_cap (fallback)
      - throughput_images_per_sec / throughput_tokens_per_sec / throughput
    """
    out = {}
    p = app_dir / "throughput.csv"
    if not p.exists():
        return out

    with p.open(newline="") as fh:
        r = csv.DictReader(fh)
        cols = set(r.fieldnames or [])

        cap_col = None
        for c in ("total_gpu_cap", "power_cap"):
            if c in cols:
                cap_col = c
                break
        if cap_col is None or "gpu_count" not in cols:
            return out

        perf_col = None
        for c in ("throughput_images_per_sec", "throughput_tokens_per_sec", "throughput"):
            if c in cols:
                perf_col = c
                break
        if perf_col is None:
            return out

        has_status = "status" in cols
        for row in r:
            if has_status and str(row.get("status", "")).strip().lower() != "ok":
                continue
            cap = f(row.get(cap_col))
            g = f(row.get("gpu_count"))
            v = f(row.get(perf_col))
            if cap is None or g is None or v is None:
                continue
            out[(int(round(cap)), int(round(g)))] = v

    return out


def extract_metrics(metric_csv):
    m = FILE_RE.match(metric_csv.name)
    if not m:
        return None
    cap = int(m.group("cap"))
    gcount = int(m.group("gpus"))

    with metric_csv.open(newline="") as fh:
        r = csv.DictReader(fh)
        rows = list(r)
        cols = r.fieldnames or []

    if not rows or "Time (s)" not in cols:
        return None

    # Drop idle samples
    if "GPU0_DRAM_Active" in cols:
        rows = [
            row for row in rows
            if (f(row.get("GPU0_DRAM_Active")) is not None and f(row.get("GPU0_DRAM_Active")) != 0.0)
        ]
    if not rows:
        return None

    def colvals(rows_in, c):
        out = []
        for row in rows_in:
            v = f(row.get(c))
            if v is not None:
                out.append(v)
        return out

    times = colvals(rows, "Time (s)")
    if not times:
        return None

    # First PROFILE_SECONDS
    t0 = min(times)
    t1 = t0 + PROFILE_SECONDS
    prof = [row for row in rows if (f(row.get("Time (s)")) is not None and f(row.get("Time (s)")) <= t1)]
    if not prof:
        prof = rows

    gpu_ids = sorted({
        int(c.split("_")[0].replace("GPU", ""))
        for c in cols if c.startswith("GPU") and "_" in c
    })
    if len(gpu_ids) >= gcount:
        gpu_ids = gpu_ids[:gcount]
    if not gpu_ids:
        return None

    p_avg, sm_avg, dr_avg = {}, {}, {}
    fp16_avg, fp32_avg, fp64_avg = {}, {}, {}

    for gid in gpu_ids:
        p_avg[gid]    = mean(colvals(prof, f"GPU{gid}_Power (W)"))
        sm_avg[gid]   = mean(colvals(prof, f"GPU{gid}_SM_Clock (MHz)"))
        dr_avg[gid]   = mean(colvals(prof, f"GPU{gid}_DRAM_Active"))
        fp16_avg[gid] = mean(colvals(prof, f"GPU{gid}_FP16_Active"))
        fp32_avg[gid] = mean(colvals(prof, f"GPU{gid}_FP32_Active"))
        fp64_avg[gid] = mean(colvals(prof, f"GPU{gid}_FP64_Active"))

    # Active GPUs
    active = [
        gid for gid in gpu_ids
        if ((not math.isnan(p_avg[gid]) and p_avg[gid] >= ACTIVE_POWER_TH) or
            (not math.isnan(sm_avg[gid]) and sm_avg[gid] >= ACTIVE_SM_TH))
    ]
    if not active:
        active = gpu_ids[:]

    n_active = len(active)

    def safe(v):
        return 0.0 if (v is None or (isinstance(v, float) and math.isnan(v))) else v

    dr_list = [safe(dr_avg[g]) for g in active]
    sm_list = [safe(sm_avg[g]) for g in active]
    fp16_list = [safe(fp16_avg[g]) for g in active]
    fp32_list = [safe(fp32_avg[g]) for g in active]
    fp64_list = [safe(fp64_avg[g]) for g in active]
    fp_list = [fp16_list[i] + fp32_list[i] + fp64_list[i] for i in range(n_active)]

    # Sum across active GPUs
    dram_active_sum = sum(dr_list)
    sm_active_sum = sum(sm_list)
    fp_all_sum = sum(fp_list)

    # Min(active) * num_active
    dram_active_min_scaled = (min(dr_list) * n_active) if n_active else 0.0
    sm_active_min_scaled = (min(sm_list) * n_active) if n_active else 0.0
    fp_all_min_scaled = (min(fp_list) * n_active) if n_active else 0.0

    return {
        "power_cap": cap,
        "gpu_count": gcount,
        "dram_active_sum": dram_active_sum,
        "sm_active_sum": sm_active_sum,
        "fp_all_sum": fp_all_sum,
        "dram_active_min_scaled": dram_active_min_scaled,
        "sm_active_min_scaled": sm_active_min_scaled,
        "fp_all_min_scaled": fp_all_min_scaled,
    }


def run_suite(root, apps, suite_name, kind):
    apps = list_apps(root, apps)
    if not apps:
        print(f"[skip] no apps in {root}")
        return

    for app in apps:
        app_dir = root / app
        if not app_dir.exists():
            continue

        if kind == "runtime":
            target_lookup = read_runtime_lookup_spec(app_dir)
            better = "lower"
        else:
            target_lookup = read_throughput_lookup_ml(app_dir)
            better = "higher"

        by_cap = defaultdict(list)

        for p in sorted(app_dir.iterdir()):
            if not FILE_RE.match(p.name):
                continue

            x = extract_metrics(p)
            if x is None:
                continue

            key = (x["power_cap"], x["gpu_count"])
            target = target_lookup.get(key, None)
            if target is None:
                continue

            x["performance"] = target
            by_cap[x["power_cap"]].append(x)

        if not by_cap:
            continue

        print(f"\n===== {suite_name}/{app} =====")
        print(f"(performance better when {better})")

        for cap in sorted(by_cap.keys()):
            rows = by_cap[cap]
            if kind == "runtime":
                rank_rows = sorted(rows, key=lambda r: r["performance"])  # lower runtime is better
            else:
                rank_rows = sorted(rows, key=lambda r: r["performance"], reverse=True)  # higher throughput is better

            true_rank = [r["gpu_count"] for r in rank_rows]
            print(f"\ncap={cap}\ttrue_rank={true_rank}")
            print("gpu_count\tperformance\tdram_sum\tsm_sum\tfp_sum\tdram_min_scaled\tsm_min_scaled\tfp_min_scaled")
            for r in rank_rows:
                print(
                    f"{r['gpu_count']}\t{fmt2(r['performance'])}\t"
                    f"{fmt2(r['dram_active_sum'])}\t{fmt2(r['sm_active_sum'])}\t{fmt2(r['fp_all_sum'])}\t"
                    f"{fmt2(r['dram_active_min_scaled'])}\t{fmt2(r['sm_active_min_scaled'])}\t{fmt2(r['fp_all_min_scaled'])}"
                )


# Capture output and save txt, while still showing in notebook
buf = StringIO()
with redirect_stdout(buf):
    run_suite(SPEC_ROOT_DIR, SPEC_APPS, "SPEC", "runtime")
    run_suite(ML_ROOT_DIR, ML_APPS, "ML", "throughput")

# text = buf.getvalue()
# TXT_OUT_PATH.write_text(text)

# print(text, end="")
# print("\n[written]", TXT_OUT_PATH)

# Unsupervised Pair Ranking

In [ ]:
# Unsupervised per-case pairwise ranking (SPEC + ML), using scaled metrics only
import re
import numpy as np
from pathlib import Path
from collections import defaultdict

TXT_PATH = Path("/home/ac.zzheng/power/GPGPU/data/H100/h100_rank_results.txt")

FEATURE_KEYS = ["dram_min_scaled", "sm_min_scaled", "fp_min_scaled"]

# Global/default seed for ML
SEED_W_SPEC = np.array([0.75, 0.15, 0.10], dtype=float)
SEED_W_ML   = np.array([0.75, 0.15, 0.10], dtype=float)

# Profiling-detected overhead-dominated regime seed (formerly VGG-like behavior)
SEED_W_OVERHEAD = np.array([1.2, -0.6, -1.0], dtype=float)

SEED_B = 0.0

assert len(SEED_W_SPEC) == len(FEATURE_KEYS)
assert len(SEED_W_ML) == len(FEATURE_KEYS)
assert len(SEED_W_OVERHEAD) == len(FEATURE_KEYS)

LR = 0.03
EPOCHS = 800
L2_SEED = 0.10
TIE_THRESH = 0.03

# Interpretable profiling-only gate thresholds
GATE_CFG = {
    "rho_sm_min": 0.8,      # Spearman(gpu_count, sm_min_scaled)
    "rho_dram_min": 0.5,    # Spearman(gpu_count, dram_min_scaled)
    "fp_flat_max": 1.25,    # max(fp_min_scaled)/min(fp_min_scaled)
    "sm_spread_min": 2.0,   # max(sm_min_scaled)/min(sm_min_scaled)
}


def sigmoid(z):
    z = np.clip(z, -40, 40)
    return 1.0 / (1.0 + np.exp(-z))


def parse_txt_9col(path):
    lines = path.read_text().splitlines()
    rows = []
    suite = app = better = None
    cap = None
    in_table = False

    header = (
        "gpu_count	performance	dram_sum	sm_sum	fp_sum	"
        "dram_min_scaled	sm_min_scaled	fp_min_scaled"
    )

    for ln in lines:
        if ln.startswith("===== ") and ln.endswith(" ====="):
            m = re.match(r"^=====\s+(\w+)/(.*?)\s+=====$", ln)
            if m:
                suite = m.group(1).strip().upper()
                app = m.group(2).strip()
            continue

        if ln.startswith("(performance better when "):
            better = "lower" if "lower" in ln else "higher"
            continue

        m = re.match(r"^cap=(\d+)\s+true_rank=\[(.*?)\]$", ln)
        if m:
            cap = int(m.group(1))
            in_table = False
            continue

        if ln.startswith(header):
            in_table = True
            continue

        if in_table and ln.strip():
            ps = ln.split("	")
            if len(ps) == 8 and ps[0].isdigit():
                rows.append({
                    "suite": suite,
                    "app": app,
                    "cap": cap,
                    "gpu": int(ps[0]),
                    "perf": float(ps[1]),
                    "dram_sum": float(ps[2]),
                    "sm_sum": float(ps[3]),
                    "fp_sum": float(ps[4]),
                    "dram_min_scaled": float(ps[5]),
                    "sm_min_scaled": float(ps[6]),
                    "fp_min_scaled": float(ps[7]),
                    "better": better,
                })
            else:
                in_table = False

    by_case = defaultdict(list)
    for r in rows:
        by_case[(r["suite"], r["app"], r["cap"])].append(r)
    return by_case


def minmax_norm(vals):
    lo, hi = min(vals), max(vals)
    if hi - lo < 1e-12:
        return [0.0] * len(vals)
    return [(v - lo) / (hi - lo) for v in vals]


def build_case_feats(rows):
    feat_cols = [minmax_norm([r[k] for r in rows]) for k in FEATURE_KEYS]
    feats = []
    for i, r in enumerate(rows):
        vec = np.array([feat_cols[j][i] for j in range(len(FEATURE_KEYS))], dtype=float)
        feats.append({
            "gpu": r["gpu"],
            "perf": r["perf"],
            "feat": vec,
            "better": r["better"],
            # raw profiling metrics used by gate (no app id)
            "dram_min_scaled": r["dram_min_scaled"],
            "sm_min_scaled": r["sm_min_scaled"],
            "fp_min_scaled": r["fp_min_scaled"],
        })
    return feats


def rankdata_avg_tie(vals):
    vals = np.asarray(vals, dtype=float)
    order = np.argsort(vals)
    ranks = np.empty(len(vals), dtype=float)
    i = 0
    while i < len(vals):
        j = i
        while j + 1 < len(vals) and vals[order[j + 1]] == vals[order[i]]:
            j += 1
        avg_rank = (i + j) / 2.0 + 1.0
        ranks[order[i:j + 1]] = avg_rank
        i = j + 1
    return ranks


def spearman_corr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 2:
        return 0.0
    if np.allclose(x, x[0]) or np.allclose(y, y[0]):
        return 0.0
    rx = rankdata_avg_tie(x)
    ry = rankdata_avg_tie(y)
    rx = rx - rx.mean()
    ry = ry - ry.mean()
    den = np.sqrt((rx * rx).sum() * (ry * ry).sum())
    if den <= 1e-12:
        return 0.0
    return float((rx * ry).sum() / den)


def safe_ratio_max_min(vals):
    vals = np.asarray(vals, dtype=float)
    if len(vals) == 0:
        return 1.0
    vmin = float(vals.min())
    vmax = float(vals.max())
    if vmin <= 1e-12:
        return float("inf") if vmax > 1e-12 else 1.0
    return vmax / vmin


def detect_overhead_regime(feats, cfg=GATE_CFG):
    g = np.array([x["gpu"] for x in feats], dtype=float)
    sm = np.array([x["sm_min_scaled"] for x in feats], dtype=float)
    dram = np.array([x["dram_min_scaled"] for x in feats], dtype=float)
    fp = np.array([x["fp_min_scaled"] for x in feats], dtype=float)

    rho_sm = spearman_corr(g, sm)
    rho_dram = spearman_corr(g, dram)
    fp_flat = safe_ratio_max_min(fp)
    sm_spread = safe_ratio_max_min(sm)

    is_overhead = (
        rho_sm >= cfg["rho_sm_min"] and
        rho_dram >= cfg["rho_dram_min"] and
        fp_flat <= cfg["fp_flat_max"] and
        sm_spread >= cfg["sm_spread_min"]
    )

    stats = {
        "rho_sm": rho_sm,
        "rho_dram": rho_dram,
        "fp_flat": fp_flat,
        "sm_spread": sm_spread,
    }
    return is_overhead, stats


def choose_seed_by_profile(feats, seed_default, seed_overhead):
    is_overhead, stats = detect_overhead_regime(feats)
    return (seed_overhead if is_overhead else seed_default), is_overhead, stats


def build_pseudo_pairs(feats, seed_w, seed_b):
    X, y = [], []
    seed_scores = [float(seed_w @ x["feat"] + seed_b) for x in feats]
    for i in range(len(feats)):
        for j in range(i + 1, len(feats)):
            diff = feats[i]["feat"] - feats[j]["feat"]
            yi = 1.0 if seed_scores[i] > seed_scores[j] else 0.0
            X.append(diff);   y.append(yi)
            X.append(-diff);  y.append(1.0 - yi)
    return np.array(X, dtype=float), np.array(y, dtype=float)


def train_case_unsup(X, y, w0, b0=0.0, lr=0.03, epochs=800, l2_seed=0.10):
    w = np.array(w0, dtype=float).copy()
    b = float(b0)
    n = len(y)
    if n == 0:
        return w, b
    for _ in range(epochs):
        p = sigmoid(X @ w + b)
        grad_w = (X.T @ (p - y)) / n + l2_seed * (w - w0)
        grad_b = np.mean(p - y)
        w -= lr * grad_w
        b -= lr * grad_b
    return w, b


def rank_case(feats, w, b):
    scores = np.zeros(len(feats), dtype=float)
    for i in range(len(feats)):
        for j in range(len(feats)):
            if i == j:
                continue
            scores[i] += sigmoid((feats[i]["feat"] - feats[j]["feat"]) @ w + b)
    order = np.argsort(-scores)
    return [feats[k]["gpu"] for k in order]


def true_rank(feats, better):
    if better == "lower":
        idx = sorted(range(len(feats)), key=lambda i: feats[i]["perf"])
    else:
        idx = sorted(range(len(feats)), key=lambda i: feats[i]["perf"], reverse=True)
    return [feats[i]["gpu"] for i in idx]


def top1_true_set(feats, better, tie_thresh=0.02):
    if not feats:
        return set()

    if better == "lower":
        ranked = sorted(feats, key=lambda x: x["perf"])
        best = ranked[0]["perf"]
        if len(ranked) == 1 or best <= 0:
            return {ranked[0]["gpu"]}
        second = ranked[1]["perf"]
        rel_gap = (second - best) / best
        return {ranked[0]["gpu"], ranked[1]["gpu"]} if rel_gap <= tie_thresh else {ranked[0]["gpu"]}

    ranked = sorted(feats, key=lambda x: x["perf"], reverse=True)
    best = ranked[0]["perf"]
    if len(ranked) == 1 or best <= 0:
        return {ranked[0]["gpu"]}
    second = ranked[1]["perf"]
    rel_gap = (best - second) / best
    return {ranked[0]["gpu"], ranked[1]["gpu"]} if rel_gap <= tie_thresh else {ranked[0]["gpu"]}


def run_eval(target_suite, by_case, seed_w_default, seed_w_overhead=None, use_profile_gate=False):
    cases = [(k, v) for k, v in sorted(by_case.items()) if k[0] == target_suite]
    print(f"
{target_suite} cases:", len(cases))
    print("FEATURE_KEYS:", FEATURE_KEYS)
    print(f"SEED_W_{target_suite}_DEFAULT:", seed_w_default.tolist())
    if seed_w_overhead is not None:
        print("SEED_W_OVERHEAD:", seed_w_overhead.tolist())
    if use_profile_gate:
        print("PROFILE_GATE_CFG:", GATE_CFG)

    hits = 0
    results = []
    weights = []

    overhead_hits = 0
    overhead_total = 0

    for (suite, app, cap), rows in cases:
        feats = build_case_feats(rows)

        used_overhead = False
        gate_stats = None
        seed_w = np.array(seed_w_default, dtype=float)
        if use_profile_gate:
            seed_w, used_overhead, gate_stats = choose_seed_by_profile(
                feats,
                seed_default=np.array(seed_w_default, dtype=float),
                seed_overhead=np.array(seed_w_overhead, dtype=float),
            )
            overhead_total += int(used_overhead)

        X, y = build_pseudo_pairs(feats, seed_w, SEED_B)

        w_case, b_case = train_case_unsup(
            X, y, w0=seed_w, b0=SEED_B, lr=LR, epochs=EPOCHS, l2_seed=L2_SEED
        )

        pred = rank_case(feats, w_case, b_case)
        better = feats[0]["better"] if feats else "lower"
        trank = true_rank(feats, better)
        tset = top1_true_set(feats, better, tie_thresh=TIE_THRESH)
        hit = int(pred[0] in tset)

        if used_overhead:
            overhead_hits += hit

        hits += hit
        results.append((app, cap, pred, trank, sorted(tset), hit, used_overhead, gate_stats))
        weights.append(w_case)

        if app.lower() == "vgg16":
            w_str = ", ".join([f"{k}={w_case[i]:.6f}" for i, k in enumerate(FEATURE_KEYS)])
            gate_tag = "OVERHEAD" if used_overhead else "DEFAULT"
            if gate_stats is not None:
                print(
                    f"[VGG16] cap={cap} gate={gate_tag} "
                    f"rho_sm={gate_stats['rho_sm']:.3f} rho_dram={gate_stats['rho_dram']:.3f} "
                    f"fp_flat={gate_stats['fp_flat']:.3f} sm_spread={gate_stats['sm_spread']:.3f} "
                    f"learned_w: {w_str}  b={b_case:.6f}"
                )
            else:
                print(f"[VGG16] cap={cap} learned_w: {w_str}  b={b_case:.6f}")

    acc = hits / len(cases) if cases else 0.0
    print(f"
Top-1 accuracy ({target_suite}): {acc:.4f} ({hits}/{len(cases)})")

    if use_profile_gate:
        print(f"Gate selected overhead seed on {overhead_total}/{len(cases)} cases")
        if overhead_total > 0:
            print(f"Top-1 on gated-overhead cases: {overhead_hits/overhead_total:.4f} ({overhead_hits}/{overhead_total})")

    print("
Average learned per-case weights:")
    if len(weights) == 0:
        print(f"No parsed {target_suite} cases. Check txt format/header.")
    else:
        W = np.vstack(weights)
        for i, k in enumerate(FEATURE_KEYS):
            print(f"w_{k} =", round(float(W[:, i].mean()), 6))

    print("
Pred vs True (all cases):")
    for app, cap, pred, trank, tset, hit, used_overhead, gate_stats in results:
        gate_tag = "OVERHEAD" if used_overhead else "DEFAULT"
        if use_profile_gate:
            print(f"{app:12s} cap={cap:4d} gate={gate_tag:8s} pred_rank={pred} true_rank={trank}")
        else:
            print(f"{app:12s} cap={cap:4d} pred_rank={pred} true_rank={trank}")

    print("
Mispredictions:")
    for app, cap, pred, trank, tset, hit, used_overhead, gate_stats in results:
        if not hit:
            gate_tag = "OVERHEAD" if used_overhead else "DEFAULT"
            if use_profile_gate:
                print(f"{app:12s} cap={cap:4d} gate={gate_tag:8s} pred_rank={pred} true_rank={trank}")
            else:
                print(f"{app:12s} cap={cap:4d} pred_rank={pred} true_rank={trank}")


# ---------- main ----------
by_case = parse_txt_9col(TXT_PATH)
print("Parsed total cases:", len(by_case))

# run_eval("SPEC", by_case, SEED_W_SPEC)
print("
=== Baseline (single seed) ===")
run_eval("ML", by_case, SEED_W_ML, use_profile_gate=False)

print("
=== Profiling-gated two-seed model ===")
run_eval("ML", by_case, SEED_W_ML, seed_w_overhead=SEED_W_OVERHEAD, use_profile_gate=True)


In [ ]:
# Unsupervised per-case pairwise ranking (SPEC + ML), using scaled metrics only
import re
import numpy as np
from pathlib import Path
from collections import defaultdict

TXT_PATH = Path("/home/ac.zzheng/power/GPGPU/data/H100/h100_rank_results.txt")

FEATURE_KEYS = ["dram_min_scaled", "sm_min_scaled", "fp_min_scaled"]

SEED_W_SPEC = np.array([0.75, 0.15, 0.10], dtype=float)
# SEED_W_ML   = np.array([0.75, 0.15, 0.10], dtype=float)
SEED_W_ML   = np.array([1.2, -0.6, -1.0], dtype=float)
SEED_B = 0.0

assert len(SEED_W_SPEC) == len(FEATURE_KEYS)
assert len(SEED_W_ML) == len(FEATURE_KEYS)

LR = 0.03
EPOCHS = 800
L2_SEED = 0.10
TIE_THRESH = 0.03


def sigmoid(z):
    z = np.clip(z, -40, 40)
    return 1.0 / (1.0 + np.exp(-z))


def parse_txt_9col(path):
    lines = path.read_text().splitlines()
    rows = []
    suite = app = better = None
    cap = None
    in_table = False

    header = (
        "gpu_count\tperformance\tdram_sum\tsm_sum\tfp_sum\t"
        "dram_min_scaled\tsm_min_scaled\tfp_min_scaled"
    )

    for ln in lines:
        if ln.startswith("===== ") and ln.endswith(" ====="):
            m = re.match(r"^=====\s+(\w+)/(.*?)\s+=====$", ln)
            if m:
                suite = m.group(1).strip().upper()
                app = m.group(2).strip()
            continue

        if ln.startswith("(performance better when "):
            better = "lower" if "lower" in ln else "higher"
            continue

        m = re.match(r"^cap=(\d+)\s+true_rank=\[(.*?)\]$", ln)
        if m:
            cap = int(m.group(1))
            in_table = False
            continue

        if ln.startswith(header):
            in_table = True
            continue

        if in_table and ln.strip():
            ps = ln.split("\t")
            if len(ps) == 8 and ps[0].isdigit():
                rows.append({
                    "suite": suite,
                    "app": app,
                    "cap": cap,
                    "gpu": int(ps[0]),
                    "perf": float(ps[1]),
                    "dram_sum": float(ps[2]),
                    "sm_sum": float(ps[3]),
                    "fp_sum": float(ps[4]),
                    "dram_min_scaled": float(ps[5]),
                    "sm_min_scaled": float(ps[6]),
                    "fp_min_scaled": float(ps[7]),
                    "better": better,
                })
            else:
                in_table = False

    by_case = defaultdict(list)
    for r in rows:
        by_case[(r["suite"], r["app"], r["cap"])].append(r)
    return by_case


def minmax_norm(vals):
    lo, hi = min(vals), max(vals)
    if hi - lo < 1e-12:
        return [0.0] * len(vals)
    return [(v - lo) / (hi - lo) for v in vals]


def build_case_feats(rows):
    feat_cols = [minmax_norm([r[k] for r in rows]) for k in FEATURE_KEYS]
    feats = []
    for i, r in enumerate(rows):
        vec = np.array([feat_cols[j][i] for j in range(len(FEATURE_KEYS))], dtype=float)
        feats.append({
            "gpu": r["gpu"],
            "perf": r["perf"],
            "feat": vec,
            "better": r["better"],
        })
    return feats


def build_pseudo_pairs(feats, seed_w, seed_b):
    X, y = [], []
    seed_scores = [float(seed_w @ x["feat"] + seed_b) for x in feats]
    for i in range(len(feats)):
        for j in range(i + 1, len(feats)):
            diff = feats[i]["feat"] - feats[j]["feat"]
            yi = 1.0 if seed_scores[i] > seed_scores[j] else 0.0
            X.append(diff);   y.append(yi)
            X.append(-diff);  y.append(1.0 - yi)
    return np.array(X, dtype=float), np.array(y, dtype=float)


def train_case_unsup(X, y, w0, b0=0.0, lr=0.03, epochs=800, l2_seed=0.10):
    w = np.array(w0, dtype=float).copy()
    b = float(b0)
    n = len(y)
    if n == 0:
        return w, b
    for _ in range(epochs):
        p = sigmoid(X @ w + b)
        grad_w = (X.T @ (p - y)) / n + l2_seed * (w - w0)
        grad_b = np.mean(p - y)
        w -= lr * grad_w
        b -= lr * grad_b
    return w, b


def rank_case(feats, w, b):
    scores = np.zeros(len(feats), dtype=float)
    for i in range(len(feats)):
        for j in range(len(feats)):
            if i == j:
                continue
            scores[i] += sigmoid((feats[i]["feat"] - feats[j]["feat"]) @ w + b)
    order = np.argsort(-scores)
    return [feats[k]["gpu"] for k in order]


def true_rank(feats, better):
    if better == "lower":
        idx = sorted(range(len(feats)), key=lambda i: feats[i]["perf"])
    else:
        idx = sorted(range(len(feats)), key=lambda i: feats[i]["perf"], reverse=True)
    return [feats[i]["gpu"] for i in idx]


def top1_true_set(feats, better, tie_thresh=0.02):
    if not feats:
        return set()

    if better == "lower":
        ranked = sorted(feats, key=lambda x: x["perf"])
        best = ranked[0]["perf"]
        if len(ranked) == 1 or best <= 0:
            return {ranked[0]["gpu"]}
        second = ranked[1]["perf"]
        rel_gap = (second - best) / best
        return {ranked[0]["gpu"], ranked[1]["gpu"]} if rel_gap <= tie_thresh else {ranked[0]["gpu"]}

    ranked = sorted(feats, key=lambda x: x["perf"], reverse=True)
    best = ranked[0]["perf"]
    if len(ranked) == 1 or best <= 0:
        return {ranked[0]["gpu"]}
    second = ranked[1]["perf"]
    rel_gap = (best - second) / best
    return {ranked[0]["gpu"], ranked[1]["gpu"]} if rel_gap <= tie_thresh else {ranked[0]["gpu"]}


def run_eval(target_suite, by_case, seed_w):
    cases = [(k, v) for k, v in sorted(by_case.items()) if k[0] == target_suite]
    print(f"\n{target_suite} cases:", len(cases))
    print("FEATURE_KEYS:", FEATURE_KEYS)
    print(f"SEED_W_{target_suite}:", seed_w.tolist())

    hits = 0
    results = []
    weights = []

    for (suite, app, cap), rows in cases:
        feats = build_case_feats(rows)
        X, y = build_pseudo_pairs(feats, seed_w, SEED_B)

        w_case, b_case = train_case_unsup(
            X, y, w0=seed_w, b0=SEED_B, lr=LR, epochs=EPOCHS, l2_seed=L2_SEED
        )


        pred = rank_case(feats, w_case, b_case)
        better = feats[0]["better"] if feats else "lower"
        trank = true_rank(feats, better)
        tset = top1_true_set(feats, better, tie_thresh=TIE_THRESH)
        hit = int(pred[0] in tset)

        hits += hit
        results.append((app, cap, pred, trank, sorted(tset), hit))
        weights.append(w_case)

    acc = hits / len(cases) if cases else 0.0
    print(f"\nTop-1 accuracy ({target_suite}): {acc:.4f} ({hits}/{len(cases)})")

    print("\nAverage learned per-case weights:")
    if len(weights) == 0:
        print(f"No parsed {target_suite} cases. Check txt format/header.")
    else:
        W = np.vstack(weights)
        for i, k in enumerate(FEATURE_KEYS):
            print(f"w_{k} =", round(float(W[:, i].mean()), 6))

    print("\nPred vs True (all cases):")
    for app, cap, pred, trank, tset, hit in results[0:0]:
        print(f"{app:12s} cap={cap:4d} pred_rank={pred} true_rank={trank}")

    print("\nMispredictions:")
    for app, cap, pred, trank, tset, hit in results:
        if not hit:
            print(f"{app:12s} cap={cap:4d} pred_rank={pred} true_rank={trank}")



# ---------- main ----------
by_case = parse_txt_9col(TXT_PATH)
print("Parsed total cases:", len(by_case))

# run_eval("SPEC", by_case, SEED_W_SPEC)
run_eval("ML", by_case, SEED_W_ML)

Parsed total cases: 135

ML cases: 45
FEATURE_KEYS: ['dram_min_scaled', 'sm_min_scaled', 'fp_min_scaled']
SEED_W_ML: [1.2, -0.6, -1.0]
[VGG16] cap=400 learned_w: dram_min_scaled=1.463156, sm_min_scaled=-0.863156, fp_min_scaled=-1.263156  b=-0.000000
[VGG16] cap=500 learned_w: dram_min_scaled=1.732480, sm_min_scaled=-1.132480, fp_min_scaled=-1.000000  b=0.000000
[VGG16] cap=600 learned_w: dram_min_scaled=1.618104, sm_min_scaled=-1.018679, fp_min_scaled=-1.443697  b=-0.000000
[VGG16] cap=700 learned_w: dram_min_scaled=1.628046, sm_min_scaled=-1.386207, fp_min_scaled=-1.000000  b=0.000000
[VGG16] cap=800 learned_w: dram_min_scaled=1.680016, sm_min_scaled=-1.395127, fp_min_scaled=-1.290939  b=-0.000000
[VGG16] cap=900 learned_w: dram_min_scaled=1.828651, sm_min_scaled=-1.280204, fp_min_scaled=-1.120094  b=0.000000
[VGG16] cap=1000 learned_w: dram_min_scaled=1.474165, sm_min_scaled=-1.607543, fp_min_scaled=-1.276607  b=-0.000000
[VGG16] cap=1100 learned_w: dram_min_scaled=1.552916, sm_min_s